# 7단계 수정판: 에이전트 루프 (6-B 방식)

`dummy_agent_ollama.ipynb` 6단계에서 확인한 것: 우리 환경(Ollama + `qwen3.5:9b`)에서는
- **6-A**(관찰을 assistant 칸에 이어 붙이기, 원문 방식) → 잘 안 됨
- **6-B**(관찰을 **새 user 메시지**로 보내기) → `Final Answer`가 깔끔하게 나옴

그런데 원래 노트북의 7단계 루프는 6-A 방식으로 짜여 있어요. 그래서 **6-B 방식으로 고친 루프**를 이 파일에 따로 만들었어요.
(원래 노트북을 고치지 않은 이유: 내 PC에서 실행하면서 결과가 파일에 저장돼 있어서, 같은 파일을 고쳐 올리면 `git pull`이 충돌해요.)

이 노트북은 혼자서도 돌아가요. 위에서부터 `Shift + Enter`.

## 준비: 0~2단계 한 번에

원래 노트북의 0단계(설정), 1단계(`chat` 함수), 2단계(`SYSTEM_PROMPT`)를 한 칸에 모았어요. 이미 아는 내용이라 실행만 하면 돼요.

In [ ]:
import json
import urllib.request

OLLAMA_URL = "http://localhost:11434"
MODEL = "qwen3.5:9b"

with urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout=10) as resp:
    models = [m["name"] for m in json.loads(resp.read())["models"]]

print("설치된 모델:", models)
print("qwen3.5:9b 있음?", MODEL in models)


def chat(messages, max_tokens=200, stop=None):
    options = {"num_predict": max_tokens}
    if stop:
        options["stop"] = stop
    body = {
        "model": MODEL,
        "messages": messages,
        "stream": False,   # 다 쓴 뒤 한 번에 받기
        "think": False,    # 속생각 끄기
        "options": options,
    }
    req = urllib.request.Request(
        f"{OLLAMA_URL}/api/chat",
        data=json.dumps(body).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=300) as resp:
        return json.loads(resp.read())["message"]["content"]


SYSTEM_PROMPT = """Answer the following questions as best you can. You have access to the following tools:

get_weather: Get the current weather in a given location

The way you use the tools is by specifying a json blob.
Specifically, this json should have an `action` key (with the name of the tool to use) and an `action_input` key (with the input to the tool going here).

The only values that should be in the "action" field are:
get_weather: Get the current weather in a given location, args: {"location": {"type": "string"}}
example use :

{{
  "action": "get_weather",
  "action_input": {"location": "New York"}
}}

ALWAYS use the following format:

Question: the input question you must answer
Thought: you should always think about one action to take. Only one action at a time in this format:
Action:

$JSON_BLOB (inside markdown cell)

Observation: the result of the action. This Observation is unique, complete, and the source of truth.
... (this Thought/Action/Observation can repeat N times, you should take several steps when needed. The $JSON_BLOB must be formatted as markdown and only use a SINGLE action at a time.)

You must always end your output with the following format:

Thought: I now know the final answer
Final Answer: the final answer to the original input question

Now begin! Reminder to ALWAYS use the exact characters `Final Answer:` when you provide a definitive answer. """


print("준비 끝:", MODEL)

✅ **확인할 것**: `준비 끝: qwen3.5:9b`가 나오면 통과. (에러가 나면 Ollama 앱이 켜져 있는지 확인)

## 루프: 손으로 한 3~6단계를 `while`처럼 반복

| 원래 노트북 | 여기 루프 안에서 |
|---|---|
| 4단계 `stop=["Observation:"]` | `chat(..., stop=["Observation:"])` |
| 5단계 JSON 꺼내기 → 도구 실행 | `json.loads(...)` → `TOOLS[...](...)` |
| 6-B 관찰을 user 메시지로 | `messages.append({"role": "user", "content": "Observation: ..."})` |
| (사람이 다시 실행) | `for`가 자동으로 다음 바퀴 |
| 끝 | `Final Answer:`가 나오면 `return` |

도시마다 다른 가짜 날씨를 써서, 여러 번 부르는 게 보이게 했어요.

In [ ]:
FAKE_WEATHER = {
    "London": "sunny with low temperatures, 8°C",
    "Paris": "rainy, 14°C",
    "Seoul": "clear and cold, 2°C",
}

def get_weather(location):
    return FAKE_WEATHER.get(location, f"Error: unknown location '{location}'. Use a full city name.")

TOOLS = {"get_weather": get_weather}


def run_agent(question, max_steps=5):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    for step in range(1, max_steps + 1):
        out = chat(messages, max_tokens=300, stop=["Observation:"])   # Thought + Action (또는 Final Answer)
        print(f"\n===== {step}바퀴: 모델이 쓴 글 =====\n{out.strip()}")
        messages.append({"role": "assistant", "content": out})        # 모델이 쓴 글을 대화에 남김

        if "Final Answer:" in out:   # 최종 답이 나오면 끝
            return out.split("Final Answer:")[-1].strip(), messages

        try:   # Action JSON 꺼내서 도구 실행
            act = json.loads(out[out.index("{"): out.rindex("}") + 1])
            obs = TOOLS[act["action"]](**act["action_input"])
        except Exception as e:   # 형식이 틀려도 멈추지 말고, 에러를 관찰로 돌려줌 (Observe 섹션!)
            obs = f"Error: could not run the action ({e})"
        print(f"----- 에이전트가 보낸 관찰 (user 메시지) -----\nObservation: {obs}")
        messages.append({"role": "user", "content": "Observation: " + obs})   # 6-B 방식

    return "(최대 횟수 도달: 답을 못 냄)", messages


final, history = run_agent("Which is warmer right now, London or Paris?")
print("\n★ 최종 답:", final)

✅ **확인할 것**
- 1바퀴에 London(또는 Paris), 2바퀴에 다른 도시, 3바퀴쯤 `Final Answer`가 나오면 대성공 (Paris 14°C가 더 따뜻함)
- 한 바퀴에 도시 두 개를 넣으려 하거나 형식이 틀리면 → `Error:` 관찰을 받고 다음 바퀴에 고치는지 보세요
- 모델은 매번 조금씩 다르게 써요. 몇 번 더 실행해서 비교해 보세요

## 대화가 어떻게 쌓였나 보기

루프가 끝난 뒤 `messages`(대본)에 무엇이 쌓였는지 역할만 찍어 봐요. 바퀴마다 **assistant(모델의 Thought+Action) → user(관찰)**가 한 쌍씩 붙어요.

In [ ]:
for i, m in enumerate(history):
    first_line = m["content"].strip().splitlines()[0] if m["content"].strip() else ""
    print(f"{i}. [{m['role']}] {first_line[:70]}")

✅ **확인할 것**: `0. [system]` → `1. [user] Which is warmer...` → `2. [assistant] Thought...` → `3. [user] Observation...` → … → 마지막 `[assistant]`에 Final Answer.
이게 Observe 섹션의 "관찰을 대본에 덧붙여 기억을 갱신한다"를 눈으로 본 거예요.

## 더 해 볼 실험 (선택)
- `run_agent("What's the weather in Seoul?")` → 한 바퀴만에 끝나나?
- `run_agent("What's the weather in Tokyo?")` → 가짜 날씨에 없는 도시. 에러 관찰을 보고 어떻게 하나?
- `run_agent("Is it colder in Seoul or London?")`

In [ ]:
final, history = run_agent("What's the weather in Tokyo?")
print("\n★ 최종 답:", final)

## 다 했으면
루프 결과(몇 바퀴 돌았나, 최종 답, Tokyo에서 어떻게 했나)를 복사해서 AI에게 붙여 주세요 → 노트 9번에 기록해요.